# SCIKIT-LEARN

It is the main open-source Machine Learning library for Python. It provides ready-made implementations of the most commonly used algorithms:

| Category | Examples |
|---|---|
| Classification | SVM, KNN, random forest, decision tree |
| Regression | Linear Regression, Ridge |
| Clustering | KMeans |
| Preprocessing | StandardScaler, get_dummies |
| Evaluation | precision_score, confusion_matrix |


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import random
from joblib import dump, load
import numpy as np
from pathlib import Path # For path handling

# The `import *` command attempts to import everything from `sklearn.model_selection`, including the experimental `HalvingGridSearchCV` which requires a special import
# Replace: from sklearn.model_selection import *
# With only what is actually needed:
from sklearn.model_selection import train_test_split

# To avoid errors, we need to import everything explicitly
# I preferred to import everything we're going to use, rather than importing the models separately
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)



In [ ]:
from warnings import simplefilter
from sklearn.exceptions import ConvergenceWarning

# Suppress convergence warnings from models that may not fully converge
# on this small dataset (not a correctness issue, just noise in output)
simplefilter("ignore", category=ConvergenceWarning)

In [ ]:
# Typo in the function name, should be read_csv instead of read_filescsv
# Now we get the from the repositry itself
ROOT = Path.cwd().parent.parent # Where your repositpry is located

PROCESSED_DATA_PATH = str(ROOT / "data" / "processed") # folder containing the processed data

train = pd.read_csv(f"{PROCESSED_DATA_PATH}/train.csv")
test  = pd.read_csv(f"{PROCESSED_DATA_PATH}/test.csv")

# Shows the number of rows and columns in each dataset, confirming they were loaded correctly and that the compression step worked as expected
print(f"Train shape : {train.shape}")
print(f"Test shape  : {test.shape}")



# ── Final Dataset Shape ───────────────────────────────────────────────────────
# Each row in the final DataFrame represents one EEG recording:
#
#  [0..18]  → 19 compressed EEG features (one per channel)
#  [19]     → eye condition  : "open" or "closed"
#  [20]     → cognitive state: "healthy" or "alzeimer"
#
#  Train : (160, 21) → 160 samples, 19 features + 2 labels
#  Test  : (32,  21) →  32 samples, 19 features + 2 labels



In [ ]:
# Summary statistics for numeric features
train.describe()

In [ ]:
test.describe()

In [ ]:
# Quick overview of data types, value ranges, and missing values
print("=== Train Info ===")
train.info(verbose=True)

In [ ]:
print("=== Test Info ===")
test.info(verbose=True)

In [ ]:

# Confirm there are no missing values in either split
print("Missing values — train:\n", train.isnull().sum())


In [ ]:
print("Missing values — test:\n",  test.isnull().sum())

In [ ]:
# Visualise the distribution of each feature to spot skew or outliers

# Training set feature distributions
train.hist(figsize=(20, 20)) # width=20 inches, height=20 inches — large enough to fit all subplots
plt.suptitle("Train Feature Distributions", fontsize=16)
plt.tight_layout() # automatically adjusts spacing between subplots to prevent overlap
plt.show()


In [ ]:
# Test set feature distributions
# Comparing with train helps detect distribution shift —
# if shapes look very different, the split may not be representative
test.hist(figsize=(20, 20))
plt.suptitle("Test Feature Distributions", fontsize=16)
plt.tight_layout()
plt.show()

# Step 1 — Encoding Categorical Labels

The last two columns contain string labels (`'open'/'closed'` and
`'healthy'/'alzeimer'`). Classical ML models require numeric input, so we
one-hot-encode them with `pd.get_dummies`.

After encoding we create a single binary target column `alz_true`:
- `1` → Alzheimer's patient
- `0` → healthy subject


In [ ]:
def encode(df: pd.DataFrame) -> pd.DataFrame:
    """One-hot-encode categorical columns and create the binary target."""
    # drop_first=True avoids multicollinearity by dropping one dummy per group
    df = pd.get_dummies(df, drop_first=True)

    # '20_healthy' = 1 means healthy; invert it to get alz_true = 1 for Alzheimer's
    df['alz_true'] = 1 - df['20_healthy']
    df = df.drop(columns=['20_healthy'])
    return df


train = encode(train)
test  = encode(test)


In [ ]:
train

In [ ]:
test

In [ ]:
# Separate features (X) and binary label (y)
# The target 'alz_true' is always the last column after encoding
X_train, y_train = train.iloc[:, :-1].values, train.iloc[:, -1].values
X_test,  y_test  = test.iloc[:, :-1].values,  test.iloc[:, -1].values


In [ ]:
print(X_train.shape, y_train.shape)

In [ ]:
print(X_test.shape, y_test.shape)

# Step 2 — Baseline Model Comparison

We train six classifiers on the training set and evaluate each on the held-out
test set. This quick sweep identifies which model families are best suited to
the data before investing in more thorough tuning.


In [ ]:
# Suggestion of otimization and cleaner code structure: 
'''
# ── Train all six baseline models and store their accuracies in a dictionary for easy comparison
baseline_models = {
    "Logistic Regression" : LogisticRegression(),
    "SVM (RBF)"           : SVC(),
    "KNN (k=3)"           : KNeighborsClassifier(n_neighbors=3),
    "Decision Tree"       : DecisionTreeClassifier(),
    "Random Forest"       : RandomForestClassifier(),
    "Gaussian NB"         : GaussianNB(),
    "Bernoulli NB"        : BernoulliNB(),
}

baseline_scores = {}
for name, model in baseline_models.items():
    model.fit(X_train, y_train)
    acc = model.score(X_test, y_test) * 100
    baseline_scores[name] = acc
    print(f"{name:<22} → {acc:.2f} %")
    
    
# Bar chart comparing all baseline model accuracies
fig, ax = plt.subplots(figsize=(12, 5))
colors = ['steelblue', 'orange', 'green', 'red', 'purple', 'pink', 'black']
ax.bar(baseline_scores.keys(), baseline_scores.values(), color=colors)
ax.set_ylabel("Accuracy (%)")
ax.set_title("Baseline Model Comparison")
ax.set_ylim(0, 110)
for i, (k, v) in enumerate(baseline_scores.items()):
    ax.text(i, v + 1, f"{v:.1f}%", ha='center', fontsize=9)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()


'''

## LINEAR MODELS

In [ ]:
logreg = LogisticRegression()
logreg.fit(X_train, y_train)

In [ ]:
Accuracy_LR = logreg.score(X_test, y_test)
print(Accuracy_LR*100)

## NORMAL NON-LINEAR MODELS

In [ ]:
clf = SVC()
clf.fit(X_train, y_train)

In [ ]:
Accuracy_SVM = clf.score(X_test, y_test)
print(Accuracy_SVM*100)

In [ ]:
knn = KNeighborsClassifier(n_neighbors = 3)
knn.fit(X_train,y_train)

In [ ]:
Accuracy_KNN = knn.score(X_test, y_test)
print(Accuracy_KNN*100)

In [ ]:
plt.bar(['KNN','SVM(RBF)'],[Accuracy_KNN,Accuracy_SVM],color = ['orange','blue'])

## TREE MODELS

In [ ]:
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

In [ ]:
Accuracy_DT = dt.score(X_test, y_test)
print(Accuracy_DT*100)

In [ ]:
rf = RandomForestClassifier()
rf.fit(X_train,y_train)

In [ ]:
# 100% accuracy is probably for a small dataset?
Accuracy_RF = rf.score(X_test, y_test)
print(Accuracy_RF*100)

In [ ]:
plt.bar(['DECISION TREE','RANDOM FOREST'],[Accuracy_DT,Accuracy_RF],color = ['red','springgreen'])

## PROBABLITY BASED MODELS

In [ ]:
gaus = GaussianNB()
gaus.fit(X_train,y_train)

In [ ]:
Accuracy_GAUS = gaus.score(X_test, y_test)
print(Accuracy_GAUS*100)

In [ ]:
bern = BernoulliNB()
bern.fit(X_train,y_train)

In [ ]:
Accuracy_BERN = bern.score(X_test, y_test)
print(Accuracy_BERN*100)

In [ ]:
plt.bar(['GAUSSIAN NB','BERNOULLI NB'],[Accuracy_GAUS,Accuracy_BERN],color = ['pink','black'])

**Based on the above results and taking into account the low sample size of the dataset, its important to train and test these models on different sets of training and testing sets to understand the generalization ability of our model**.
# Step 3 — Comparison of Various ML models on Different Sets of Data


*   Our comparison technique starts by splitting a whole dataset into train/test in 15 random ways using sklearn parameter "random_state".
*   In our next step fit the best model from groups (Non-linear, Tree since they are much better) on these 15 different datasets and note down the model's score on the randomly created on every test set.
*   Next we plot the results to find which model has the highest accuracy and at which "random_state" did we obtain it.
*   Then we take the models and fit by the best "random_state" to build a combined classifier to combine all the best models as our finalized model.
*   The best "random_state" can be obtained by finding which "random_state" produces most accuracy for the model.



## Why Test on 15 Different Splits?

With only 192 samples total, a single train/test split is unreliable.
Depending on which samples end up in the test set by chance, the same model
can score anywhere from 60% to 95% — making it impossible to know if a good
result is genuine or just lucky.

**The solution:** split the data 15 times using different random seeds and
measure the model's accuracy on each split. This gives us a distribution of
scores instead of a single number, revealing:
- How **consistent** the model is across different data arrangements
- The **best random_state** — the split where the model performed best

### Workflow
1. Combine train and test back into one pool
2. Evaluate the 3 best models (SVM, KNN, Random Forest) across 15 random splits
3. Plot the accuracy curves to visually compare consistency
4. Identify the best random_state per model (peak of each curve)
5. Retrain each model using its best random_state to build the final combined classifier

> A model that scores consistently high across all 15 splits is genuinely
> learning the pattern — not just getting lucky with one particular split.

## i) Defining Functions for comparison

In [ ]:
X = np.concatenate([X_train,X_test])
y = np.concatenate([y_train,y_test])

In [ ]:
color = ['blue','springgreen','yellow']

In [ ]:
# Generate 15 random states once so every model is evaluated on the exact
# same 15 splits — ensuring a fair comparison between models
rands = [random.randint(1, 300) for i in range(15)]

def fit_and_score(model):
    """Train and evaluate a model across 15 different random train/test splits.
    
    Returns
    -------
    scores : list[float]  Accuracy (%) on each of the 15 test sets
    states : list[int]    The random_state used for each split
    """
    scores = []
    states = []

    for i in range(15):
        state = rands[i]

        # Create a new 70/30 train/test split using a different random seed each time
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.3, random_state=state
        )

        # Train the model from scratch on this specific split
        model.fit(X_train, y_train)

        # Record the accuracy and the random state that produced it
        scores.append(model.score(X_test, y_test) * 100)
        states.append(state)

    return scores, states

In [ ]:
models = [clf,knn,rf] # Best models & will discard NB models & Logistic since they produce low perfromance score
model_names = ["SupportVectorMachine","KNearestNeighour","RandomForest"]  
plot_scores = []
r_states = []

for model in models:
    scores,states = fit_and_score(model)
    plot_scores.append(scores)
    r_states.append(states)

## ii) Plotting of all Models Performances on Different data

In [ ]:
f = plt.figure()
f.set_figwidth(20)
f.set_figheight(10)

for i in range(0,3):
    plt.plot([s for s in range(1,16)],plot_scores[i],label = model_names[i],color = color[i])
    plt.legend()
    max_ind = plot_scores[i].index(max(plot_scores[i])) 
    max_state = r_states[i][max_ind]
    print("Model : ",model_names[i]," ->  Best State : ",max_state)
    plt.text(max_ind+1,plot_scores[i][max_ind],str("State : " + str(max_state)),size = 10,color = color[i])

plt.xlabel("Different Sets")
plt.ylabel("Accuracy")
plt.show()    


# Step 4 — Buidling a combined classifier

The reason to train a combined classifier is to increase a generalization of the output so that each model from the above best models can produce its own output based on what it has learnt apart from other models and combine those outputs to provide a more assured answer. During testing we predict test cases on these best models and use majority voting (2-out-of-3) to produce the final output.

## (i) Training Phase

In [ ]:
models = [SVC(),
          KNeighborsClassifier(n_neighbors = 3),
          RandomForestClassifier()]

model_names = ["SVM","KNN","RFC"]
best_r_states = [105,105,105]
def state_scores(model,name,state):  
    X_train,X_test,y_train,y_test=train_test_split(X, y, test_size=0.3,random_state=state)
    # Adding Noise to improve as a response to small dataset
    model.fit(X_train + np.random.normal(size = X_train.shape,loc = 0,scale = 0.01), y_train)
    name = name + ".joblib"
    dump(model,name)
    return model.score(X_test,y_test)*100
scores = []
for i in range(len(models)):
    print("Model Name : ",model_names[i])
    score = state_scores(models[i],model_names[i],best_r_states[i])
    scores.append(score)
    print("Score : ",score)    

In [ ]:
plt.bar(model_names,scores,color = color)

## (i) Testing Phase

In [ ]:

def combined_classifier():
    # Load each model from the current working directory
    # FIX: original code used /content/ (Google Colab path) — replaced with relative path
    knn  = load("KNN.joblib")
    svvc = load("SVM.joblib")
    rfc  = load("RFC.joblib")
    return (svvc, knn, rfc)

models = combined_classifier()

def predict(test):
    # Collect predictions from each model independently
    result1 = models[0].predict(test)  # SVM prediction
    result2 = models[1].predict(test)  # KNN prediction
    result3 = models[2].predict(test)  # Random Forest prediction

    # Stack predictions: shape (3, n_samples) → transpose → (n_samples, 3)
    results = np.array([result1, result2, result3]).T.tolist()

    # Majority voting: for each sample pick the label that appears most (2 out of 3)
    results = [Counter(res).most_common(1)[0][0] for res in results]
    return np.array(results)

**Calling the combined classifier "predict"**

In [ ]:
preds = predict(X_test)

## Performance Metrics 

In [ ]:
print("Combined Classifier — Test Set Performance")
print("=" * 45)
print(f"  Accuracy  : {accuracy_score(y_test, preds)*100:.2f} %")
print(f"  F1 Score  : {f1_score(y_test, preds)*100:.2f} %")
print(f"  Precision : {precision_score(y_test, preds)*100:.2f} %")


In [ ]:
# Confusion matrix visualisation
# Rows = actual labels, Columns = predicted labels
cm = confusion_matrix(y_test, preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=['Healthy', 'Alzheimer'])
disp.plot(colorbar=False, cmap='binary')
plt.title("Confusion Matrix — Combined Classifier")
plt.tight_layout()
plt.show()

# Raw counts for reference
tn, fp, fn, tp = cm.ravel()
print(f"TP={tp}  FP={fp}  TN={tn}  FN={fn}")
